## Random Forest

In [1]:
pip install fastparquet

Note: you may need to restart the kernel to use updated packages.


In [24]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, classification_report
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

In [25]:
import pandas as pd
from pathlib import Path

root = next(p for p in Path.cwd().parents if (p / "config.yml").exists())

panel = pd.read_parquet(
    root / "data/processed_data/analysis_panel.parquet", engine='fastparquet'
)
print(panel.shape)
print(panel.dtypes)
panel.head(20)

(17150, 57)
YEAR                               int16
GEOGRAPHY_CODE                  category
GEOGRAPHY_NAME                  category
IS8_SECTOR                      category
EMPLOYEES                        float32
BUSINESSES                       float32
gva_per_hour                     float64
weekly_pay                       float64
employment_rate                  float64
unemployment_rate                float64
gdhi_per_head                    float64
new_enterprises                  float64
deaths_of_enterprises            float64
active_enterprises               float64
high_growth_enterprises          float64
public_transport_to_employer     float64
drive_to_employer                float64
cycle_to_employer                float64
broadband_availability           float64
4g_area_coverage                 float64
ks2_attainment                   float64
gcse_by_age_19                   float64
ofsted                           float64
persistent_absences              float64
pers

,YEAR,GEOGRAPHY_CODE,GEOGRAPHY_NAME,IS8_SECTOR,EMPLOYEES,BUSINESSES,gva_per_hour,weekly_pay,employment_rate,unemployment_rate,...,lq_bus,emp_share,lq_emp,growth_emp,cagr_emp,growth_bus,cagr_bus,related_variety,size_large_share,size_micro_share
0,2016,E06000001,Hartlepool,Advanced Manufacturing,1470.0,25.0,29.84,521.2,69.7,4.6,...,0.840582,0.048197,1.661003,0.010204,0.001693,-0.200000,-0.036508,1.332179,0.000000,1.000000
1,2016,E06000001,Hartlepool,Creative Industries,450.0,110.0,29.84,521.2,69.7,4.6,...,0.355425,0.014754,0.297588,0.744444,0.097176,-0.090909,-0.015760,2.200516,0.000000,1.000000
2,2016,E06000001,Hartlepool,Defence,0.0,0.0,29.84,521.2,69.7,4.6,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN
3,2016,E06000001,Hartlepool,Digital and Technologies,1510.0,440.0,29.84,521.2,69.7,4.6,...,1.362619,0.049508,0.701351,-0.076159,-0.013116,-0.352273,-0.069823,1.072313,0.000000,1.000000
4,2016,E06000001,Hartlepool,Financial Services,285.0,40.0,29.84,521.2,69.7,4.6,...,0.383770,0.009344,0.157782,-0.070175,-0.012053,0.375000,0.054509,1.213008,0.000000,0.750000
5,2016,E06000001,Hartlepool,Life Sciences,40.0,0.0,29.84,521.2,69.7,4.6,...,0.000000,0.001311,0.456144,0.000000,0.000000,NaN,NaN,0.000000,NaN,NaN
6,2016,E06000001,Hartlepool,Professional and Business Services,2630.0,870.0,29.84,521.2,69.7,4.6,...,0.991174,0.086230,0.483176,-0.019011,-0.003194,-0.264368,-0.049884,1.830641,0.000000,0.965517
7,2016,E06000002,Middlesbrough,Advanced Manufacturing,620.0,25.0,29.50,481.9,68.8,5.1,...,0.587201,0.010622,0.366062,0.153226,0.024045,0.400000,0.057681,1.332179,0.000000,0.800000
8,2016,E06000002,Middlesbrough,Creative Industries,1320.0,180.0,29.50,481.9,68.8,5.1,...,0.406288,0.022614,0.456128,0.431818,0.061650,0.055556,0.009052,2.328951,0.000000,1.000000
9,2016,E06000002,Middlesbrough,Defence,0.0,0.0,29.50,481.9,68.8,5.1,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN


In [26]:

# ─────────────────────────────────────────────
# 1. CARGAR DATOS
# ─────────────────────────────────────────────
root = next(p for p in Path.cwd().parents if (p / "config.yml").exists())

df = pd.read_parquet(
    root / "data/processed_data/analysis_panel.parquet", engine='fastparquet'
)
print(panel.shape)
print(panel.dtypes)
panel.head(20)

# Filtrar solo 2022
df = df[df["YEAR"] == 2022].copy()
print(f"Shape filtrado (2022): {df.shape}")
print("Sectores IS8:", df["IS8_SECTOR"].unique())

(17150, 57)
YEAR                               int16
GEOGRAPHY_CODE                  category
GEOGRAPHY_NAME                  category
IS8_SECTOR                      category
EMPLOYEES                        float32
BUSINESSES                       float32
gva_per_hour                     float64
weekly_pay                       float64
employment_rate                  float64
unemployment_rate                float64
gdhi_per_head                    float64
new_enterprises                  float64
deaths_of_enterprises            float64
active_enterprises               float64
high_growth_enterprises          float64
public_transport_to_employer     float64
drive_to_employer                float64
cycle_to_employer                float64
broadband_availability           float64
4g_area_coverage                 float64
ks2_attainment                   float64
gcse_by_age_19                   float64
ofsted                           float64
persistent_absences              float64
pers

In [34]:
# ─────────────────────────────────────────────
# DEFINIR LOS DOS GRUPOS DE FEATURES
# ─────────────────────────────────────────────
features_full = [
    "new_enterprises", "deaths_of_enterprises", "active_enterprises",
    "high_growth_enterprises", "broadband_availability", "4g_area_coverage",
    "gcse_by_age_19", "early_years_comms", "early_years_literacy",
    "early_years_maths", "apprenticeship_starts", "apprenticeship_achievements",
    "level_3+_qualifications", "fe_and_skills_participation", "smokers",
    "reception_obesity", "year_6_obesity", "adult_obesity", "cancer_diagnosis",
    "under_75_mortality_rate", "life_satisfaction", "worthwhile", "happiness",
    "anxiety", "net_additions", "related_variety", "size_large_share",
    "size_micro_share", "public_transport_to_employer", "drive_to_employer",
    "cycle_to_employer"
]

features_theory = [
    "level_3+_qualifications",       # human capital
    "gcse_by_age_19",                 # human capital
    "apprenticeship_starts",          # human capital
    "apprenticeship_achievements",    # human capital
    "fe_and_skills_participation",    # human capital
    "new_enterprises",                # entrepreneurial discovery
    "deaths_of_enterprises",          # entrepreneurial discovery
    "active_enterprises",             # entrepreneurial discovery
    "high_growth_enterprises",        # entrepreneurial discovery
    "broadband_availability",         # connectivity
    "4g_area_coverage",               # connectivity
    "net_additions",                  # place
    "related_variety",                # EEG — capability spillovers
]

feature_groups = {
    "full":   features_full,
    "theory": features_theory,
}

# ─────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────
RF_PARAMS_REG = dict(n_estimators=100, max_depth=4, min_samples_leaf=5,
                     n_jobs=-1, random_state=42)
RF_PARAMS_CLF = dict(n_estimators=100, max_depth=4, min_samples_leaf=5,
                     n_jobs=-1, random_state=42)

CV_REG = KFold(n_splits=5, shuffle=True, random_state=42)
CV_CLF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

dep_vars_regression     = ["lq_emp", "lq_bus", "cagr_emp", "cagr_bus"]
dep_vars_classification = ["lq_emp", "lq_bus", "cagr_emp", "cagr_bus"]

sectors = sorted(df["IS8_SECTOR"].dropna().unique())

# ─────────────────────────────────────────────
# LOOP PRINCIPAL — dos grupos
# ─────────────────────────────────────────────
all_results = []
all_fi      = []

for group_name, features in feature_groups.items():

    # Verificar features disponibles
    missing = [f for f in features if f not in df.columns]
    if missing:
        print(f"⚠️  [{group_name}] Columnas no encontradas: {missing}")
    features = [f for f in features if f in df.columns]

    print(f"\n{'#'*60}")
    print(f"  GRUPO: {group_name.upper()}  |  {len(features)} features")
    print(f"{'#'*60}")

    for sector in sectors:
        sub = df[df["IS8_SECTOR"] == sector].copy()
        X   = sub[features].copy()
        X   = X.fillna(X.median(numeric_only=True))

        print(f"\n{'='*60}")
        print(f"  SECTOR: {sector}  |  n = {len(sub)}")
        print(f"{'='*60}")

        # ── REGRESIÓN ────────────────────────────────────────────
        for dep in dep_vars_regression:
            if dep not in sub.columns:
                continue
            y    = sub[dep].copy()
            mask = y.notna()
            X_r, y_r = X[mask], y[mask]

            if len(y_r) < 20:
                print(f"  [SKIP regresión {dep}] n={len(y_r)} < 20")
                continue

            model = RandomForestRegressor(**RF_PARAMS_REG)
            r2_cv   = cross_val_score(model, X_r, y_r, cv=CV_REG, scoring="r2")
            rmse_cv = np.sqrt(-cross_val_score(model, X_r, y_r, cv=CV_REG,
                              scoring="neg_mean_squared_error"))

            print(f"  [Reg {group_name}] {dep} | R²={r2_cv.mean():.3f} ± {r2_cv.std():.3f}")

            # Fit final para feature importance
            model.fit(X_r, y_r)
            all_fi.append(pd.DataFrame({
                "group":      group_name,
                "sector":     sector,
                "dep_var":    dep,
                "model_type": "regression",
                "feature":    features,
                "importance": model.feature_importances_
            }))

            all_results.append({
                "group":        group_name,
                "sector":       sector,
                "dep_var":      dep,
                "model_type":   "regression",
                "n":            int(mask.sum()),
                "R2_mean":      round(r2_cv.mean(), 4),
                "R2_std":       round(r2_cv.std(),  4),
                "RMSE_mean":    round(rmse_cv.mean(), 4),
                "RMSE_std":     round(rmse_cv.std(),  4),
            })

        # ── CLASIFICACIÓN ────────────────────────────────────────
        for dep in dep_vars_classification:
            if dep not in sub.columns:
                continue
            y_cont = sub[dep].copy()
            mask   = y_cont.notna()
            X_c    = X[mask]
            threshold = 1 if dep in ("lq_emp", "lq_bus") else y_cont[mask].median()
            y_c    = (y_cont[mask] >= threshold).astype(int)
            dep_clf = f"{dep}_binary"

            if len(y_c) < 20 or y_c.nunique() < 2:
                print(f"  [SKIP clasificación {dep_clf}] n<20 o clase única")
                continue

            model_c = RandomForestClassifier(**RF_PARAMS_CLF)
            f1_cv  = cross_val_score(model_c, X_c, y_c, cv=CV_CLF, scoring="f1")
            acc_cv = cross_val_score(model_c, X_c, y_c, cv=CV_CLF, scoring="accuracy")

            print(f"  [Clf {group_name}] {dep_clf} | F1={f1_cv.mean():.3f} ± {f1_cv.std():.3f}")

            # Fit final para feature importance
            model_c.fit(X_c, y_c)
            all_fi.append(pd.DataFrame({
                "group":      group_name,
                "sector":     sector,
                "dep_var":    dep_clf,
                "model_type": "classification",
                "feature":    features,
                "importance": model_c.feature_importances_
            }))

            all_results.append({
                "group":      group_name,
                "sector":     sector,
                "dep_var":    dep_clf,
                "model_type": "classification",
                "n":          int(mask.sum()),
                "F1_mean":    round(f1_cv.mean(), 4),
                "F1_std":     round(f1_cv.std(),  4),
                "Acc_mean":   round(acc_cv.mean(), 4),
                "Acc_std":    round(acc_cv.std(),  4),
            })

# ─────────────────────────────────────────────
# EXPORTAR Y COMPARAR
# ─────────────────────────────────────────────
results_df = pd.DataFrame(all_results)
fi_df      = pd.concat(all_fi, ignore_index=True) if all_fi else pd.DataFrame()

# Exportar todo
results_df.to_csv("rf_two_groups_metrics.csv",      index=False)
fi_df.to_csv("rf_two_groups_feature_importance.csv", index=False)

# ── Tabla comparativa regresión ──
reg_df = results_df[results_df["model_type"] == "regression"]
comp_reg = reg_df.pivot_table(
    index=["sector", "dep_var"],
    columns="group",
    values=["R2_mean", "RMSE_mean"]
).round(4)
comp_reg.columns = ["_".join(c) for c in comp_reg.columns]
comp_reg["R2_winner"] = np.where(
    comp_reg["R2_mean_full"] >= comp_reg["R2_mean_theory"], "full", "theory"
)
comp_reg.to_csv("rf_comparison_regression.csv")
print("\n── COMPARACIÓN REGRESIÓN (R²) ──")
print(comp_reg[["R2_mean_full", "R2_mean_theory", "R2_winner"]].to_string())

# ── Tabla comparativa clasificación ──
clf_df = results_df[results_df["model_type"] == "classification"]
comp_clf = clf_df.pivot_table(
    index=["sector", "dep_var"],
    columns="group",
    values=["F1_mean", "Acc_mean"]
).round(4)
comp_clf.columns = ["_".join(c) for c in comp_clf.columns]
comp_clf["F1_winner"] = np.where(
    comp_clf["F1_mean_full"] >= comp_clf["F1_mean_theory"], "full", "theory"
)
comp_clf.to_csv("rf_comparison_classification.csv")
print("\n── COMPARACIÓN CLASIFICACIÓN (F1) ──")
print(comp_clf[["F1_mean_full", "F1_mean_theory", "F1_winner"]].to_string())

print("\n✅ Archivos exportados:")
print("   rf_two_groups_metrics.csv")
print("   rf_two_groups_feature_importance.csv")
print("   rf_comparison_regression.csv")
print("   rf_comparison_classification.csv")


############################################################
  GRUPO: FULL  |  31 features
############################################################

  SECTOR: Advanced Manufacturing  |  n = 350
  [Reg full] lq_emp | R²=0.235 ± 0.158
  [Reg full] lq_bus | R²=0.427 ± 0.062
  [Reg full] cagr_emp | R²=-0.079 ± 0.028
  [Reg full] cagr_bus | R²=0.094 ± 0.126
  [Clf full] lq_emp_binary | F1=0.655 ± 0.051
  [Clf full] lq_bus_binary | F1=0.754 ± 0.062
  [Clf full] cagr_emp_binary | F1=0.514 ± 0.060
  [Clf full] cagr_bus_binary | F1=0.691 ± 0.026

  SECTOR: Creative Industries  |  n = 350
  [Reg full] lq_emp | R²=0.643 ± 0.160
  [Reg full] lq_bus | R²=0.747 ± 0.068
  [Reg full] cagr_emp | R²=-0.012 ± 0.068
  [Reg full] cagr_bus | R²=0.088 ± 0.047
  [Clf full] lq_emp_binary | F1=0.562 ± 0.142
  [Clf full] lq_bus_binary | F1=0.800 ± 0.026
  [Clf full] cagr_emp_binary | F1=0.570 ± 0.050
  [Clf full] cagr_bus_binary | F1=0.573 ± 0.040

  SECTOR: Defence  |  n = 350
  [Reg full] lq_emp | R²=-30.

In [33]:
# ─────────────────────────────────────────────
# 5. EXPORTAR
# ─────────────────────────────────────────────
results_df = pd.DataFrame(results)
fi_df      = pd.concat(fi_store,   ignore_index=True) if fi_store   else pd.DataFrame()
preds_df   = pd.concat(pred_store, ignore_index=True) if pred_store else pd.DataFrame()

results_df.to_csv("rf_metrics.csv",           index=False)
fi_df.to_csv("rf_feature_importance.csv",     index=False)
preds_df.to_csv("rf_predictions.csv",         index=False)

print("\n✅ Archivos exportados.")

# ─────────────────────────────────────────────
# 6. RESUMEN — separado por tipo de modelo
# ─────────────────────────────────────────────

# Regresión: R² y RMSE
reg_df = results_df[results_df["model_type"] == "regression"][
    ["sector", "dep_var", "n", "r2_cv_mean", "r2_cv_std", "rmse_cv_mean", "rmse_cv_std"]
].copy()
reg_df.columns = ["sector", "dep_var", "n", "R2_mean", "R2_std", "RMSE_mean", "RMSE_std"]

# Clasificación: F1
clf_df = results_df[results_df["model_type"] == "classification"][
    ["sector", "dep_var", "n", "f1_cv_mean", "f1_cv_std", "acc_cv_mean", "acc_cv_std"]
].copy()
clf_df.columns = ["sector", "dep_var", "n", "F1_mean", "F1_std", "Acc_mean", "Acc_std"]

print("\n── REGRESIÓN (RF) ── comparar con Lasso por R² y RMSE ──")
print(reg_df.to_string(index=False))

print("\n── CLASIFICACIÓN (RF) ── comparar con Lasso por F1 ──")
print(clf_df.to_string(index=False))

# Exportar tablas limpias para comparación
reg_df.to_csv("rf_regression_summary.csv",       index=False)
clf_df.to_csv("rf_classification_summary.csv",   index=False)
print("\n✅ Tablas de comparación exportadas:")
print("   rf_regression_summary.csv")
print("   rf_classification_summary.csv")


✅ Archivos exportados.

── REGRESIÓN (RF) ── comparar con Lasso por R² y RMSE ──
                            sector  dep_var   n  R2_mean  R2_std  RMSE_mean  RMSE_std
            Advanced Manufacturing   lq_emp 350   0.2354  0.1575     1.1393    0.3169
            Advanced Manufacturing   lq_bus 350   0.4266  0.0617     0.3391    0.0360
            Advanced Manufacturing cagr_emp 350  -0.0788  0.0280     0.0753    0.0344
            Advanced Manufacturing cagr_bus 349   0.0937  0.1263     0.0407    0.0080
               Creative Industries   lq_emp 350   0.6433  0.1605     0.3856    0.1365
               Creative Industries   lq_bus 350   0.7470  0.0681     0.2388    0.0253
               Creative Industries cagr_emp 350  -0.0117  0.0675     0.0402    0.0047
               Creative Industries cagr_bus 349   0.0875  0.0465     0.0177    0.0027
                           Defence   lq_emp 350 -30.5146 60.4552     8.0919    4.5281
                           Defence   lq_bus 350  -0.0019  

In [35]:
import pandas as pd
df_excel = pd.read_excel("feature_importance_por_sector_full_data.xlsx")
print(df_excel.head(10).to_string())
print("\nColumnas:", df_excel.columns.tolist())
print("Shape:", df_excel.shape)

  Sector: Advanced Manufacturing Unnamed: 1 Unnamed: 2  Unnamed: 3  Unnamed: 4
0                            NaN        NaN        NaN         NaN         NaN
1                     REGRESSION        NaN        NaN         NaN         NaN
2                        Feature     lq_emp     lq_bus  growth_emp  growth_bus
3                       cagr_emp        NaN        NaN      0.9936      0.0047
4                       cagr_bus        NaN     0.0282         NaN      0.9857
5                      emp_share     0.9836     0.4644         NaN         NaN
6                     BUSINESSES        NaN     0.1873         NaN         NaN
7             active_enterprises     0.0021     0.0584      0.0009         NaN
8               size_micro_share        NaN     0.0362         NaN         NaN
9                      happiness        NaN        NaN         NaN      0.0022

Columnas: ['Sector: Advanced Manufacturing', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']
Shape: (20, 5)


In [37]:
# ─────────────────────────────────────────────
# FEATURE IMPORTANCE — formato pivotado por sector
# Una hoja por sector, columnas = dep_vars
# ─────────────────────────────────────────────
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
import pandas as pd

TOP_N = 5

dep_vars_reg = ["lq_emp", "lq_bus", "cagr_emp", "cagr_bus"]
dep_vars_clf = ["lq_emp_binary", "lq_bus_binary", "cagr_emp_binary", "cagr_bus_binary"]

def build_pivot_for_sector(group_fi, sector, dep_reg, dep_clf, top_n=5):
    """
    Construye la tabla pivotada para un sector:
    filas = top features (union de todas las dep_vars)
    columnas = dep_vars
    separadas en bloque REGRESSION y CLASSIFICATION
    """
    rows = []

    for block_label, dep_list in [("REGRESSION", dep_reg), ("CLASSIFICATION", dep_clf)]:
        # Filtrar sector y dep_vars del bloque
        block_fi = group_fi[
            (group_fi["sector"] == sector) &
            (group_fi["dep_var"].isin(dep_list))
        ]
        if block_fi.empty:
            continue

        # Top N features por dep_var
        top_features = (
            block_fi
            .sort_values("importance", ascending=False)
            .groupby("dep_var")
            .head(top_n)
        )

        # Pivot: filas=feature, columnas=dep_var, valores=importance
        pivot = top_features.pivot_table(
            index="feature",
            columns="dep_var",
            values="importance",
            aggfunc="first"
        ).reindex(columns=dep_list)  # orden fijo de columnas

        # Eliminar filas donde todos son NaN
        pivot = pivot.dropna(how="all")
        pivot = pivot.reset_index()
        pivot.columns.name = None

        # Agregar fila de encabezado del bloque
        rows.append({"block": block_label, "data": pivot})

    return rows

for group_name in ["full", "theory"]:
    group_fi = fi_df[fi_df["group"] == group_name].copy()

    with pd.ExcelWriter(f"feature_importance_{group_name}.xlsx", engine="openpyxl") as writer:

        for sector in sorted(group_fi["sector"].unique()):
            blocks = build_pivot_for_sector(
                group_fi, sector, dep_vars_reg, dep_vars_clf, TOP_N
            )
            if not blocks:
                continue

            # ── Escribir hoja ──
            sheet_name = sector[:31]
            workbook   = writer.book
            worksheet  = workbook.create_sheet(title=sheet_name)

            # Estilos
            header_font    = Font(bold=True, color="FFFFFF")
            header_fill    = PatternFill("solid", fgColor="2F5496")  # azul oscuro
            section_fill   = PatternFill("solid", fgColor="D9E1F2")  # azul claro
            center_align   = Alignment(horizontal="center")

            # Título del sector
            worksheet.cell(row=1, column=1, value=f"Sector: {sector}").font = Font(bold=True, size=12)

            current_row = 2

            for block in blocks:
                label = block["block"]
                pivot = block["data"]
                cols  = pivot.columns.tolist()  # ["feature"] + dep_vars

                # ── Fila etiqueta bloque (REGRESSION / CLASSIFICATION) ──
                cell = worksheet.cell(row=current_row, column=1, value=label)
                cell.font = Font(bold=True, color="FFFFFF")
                cell.fill = header_fill
                cell.alignment = center_align
                worksheet.merge_cells(
                    start_row=current_row, start_column=1,
                    end_row=current_row,   end_column=len(cols)
                )
                current_row += 1

                # ── Fila de encabezados de columnas ──
                for col_idx, col_name in enumerate(cols, start=1):
                    cell = worksheet.cell(row=current_row, column=col_idx, value=col_name)
                    cell.font = Font(bold=True)
                    cell.fill = section_fill
                    cell.alignment = center_align
                current_row += 1

                # ── Filas de datos ──
                for _, data_row in pivot.iterrows():
                    for col_idx, col_name in enumerate(cols, start=1):
                        val = data_row[col_name]
                        # Redondear importancias, dejar NaN como vacío
                        if col_name != "feature" and pd.notna(val):
                            val = round(float(val), 4)
                        elif pd.isna(val):
                            val = None
                        worksheet.cell(row=current_row, column=col_idx, value=val)
                    current_row += 1

                current_row += 1  # fila vacía entre bloques

            # ── Ajustar ancho de columnas ──
            for col_idx in range(1, len(dep_vars_reg) + 2):
                col_letter = get_column_letter(col_idx)
                worksheet.column_dimensions[col_letter].width = 28 if col_idx == 1 else 14

    print(f"✅ Exportado: feature_importance_{group_name}.xlsx")


✅ Exportado: feature_importance_full.xlsx
✅ Exportado: feature_importance_theory.xlsx


In [11]:
# summary for lq_emp regression

print("\n── TOP 5 FEATURES por sector (regresión lq_emp) ──────────")
top_fi_lqemp = (
    fi_df[(fi_df["dep_var"] == "lq_emp") & (fi_df["model_type"] == "regression")]
    .sort_values(["sector", "importance"], ascending=[True, False])
    .groupby("sector")
    .head(5)[["sector", "feature", "importance"]]
    .sort_values(["sector", "importance"], ascending=[True, False])
)
print(top_fi_lqemp.to_string(index=False))


── TOP 5 FEATURES por sector (regresión lq_emp) ──────────
                            sector                      feature  importance
            Advanced Manufacturing                net_additions    0.193521
            Advanced Manufacturing  apprenticeship_achievements    0.133203
            Advanced Manufacturing            unemployment_rate    0.087157
            Advanced Manufacturing        apprenticeship_starts    0.084238
            Advanced Manufacturing              related_variety    0.073376
               Creative Industries public_transport_to_employer    0.344499
               Creative Industries        apprenticeship_starts    0.193604
               Creative Industries            cycle_to_employer    0.100045
               Creative Industries      level_3+_qualifications    0.059428
               Creative Industries               gcse_by_age_19    0.049106
                           Defence       broadband_availability    0.123245
                           D

In [11]:
# summary for lq_bus regression

print("\n── TOP 5 FEATURES por sector (regression lq_bus) ──────────")
top_fi_lqbus = (
    fi_df[(fi_df["dep_var"] == "lq_bus") & (fi_df["model_type"] == "regression")]
    .sort_values(["sector", "importance"], ascending=[True, False])
    .groupby("sector")
    .head(5)[["sector", "feature", "importance"]]
    .sort_values(["sector", "importance"], ascending=[True, False])
)
print(top_fi_lqbus.to_string(index=False))


── TOP 5 FEATURES por sector (regression lq_bus) ──────────
                            sector                      feature  importance
            Advanced Manufacturing                    emp_share    0.464442
            Advanced Manufacturing                   BUSINESSES    0.187295
            Advanced Manufacturing           active_enterprises    0.058384
            Advanced Manufacturing             size_micro_share    0.036249
            Advanced Manufacturing                     cagr_bus    0.028220
               Creative Industries                    emp_share    0.595507
               Creative Industries public_transport_to_employer    0.083526
               Creative Industries                gdhi_per_head    0.060653
               Creative Industries        apprenticeship_starts    0.044552
               Creative Industries                   BUSINESSES    0.043429
                           Defence                   BUSINESSES    0.498294
                           

In [12]:
# Summary for classification - lq_emp

print("\n── TOP 5 FEATURES por sector (classification lq_emp) ──────────")
top_fi_class_lqemp = (
    fi_df[(fi_df["dep_var"] == "lq_emp_binary") & (fi_df["model_type"] == "classification")]
    .sort_values(["sector", "importance"], ascending=[True, False])
    .groupby("sector")
    .head(5)[["sector", "feature", "importance"]]
    .sort_values(["sector", "importance"], ascending=[True, False])
)
print(top_fi_class_lqemp.to_string(index=False))


── TOP 5 FEATURES por sector (classification lq_emp) ──────────
                            sector                     feature  importance
            Advanced Manufacturing                   emp_share    0.480121
            Advanced Manufacturing                   EMPLOYEES    0.131404
            Advanced Manufacturing            size_micro_share    0.033812
            Advanced Manufacturing apprenticeship_achievements    0.027540
            Advanced Manufacturing       apprenticeship_starts    0.024995
               Creative Industries                   emp_share    0.316592
               Creative Industries                   EMPLOYEES    0.093773
               Creative Industries               gdhi_per_head    0.075466
               Creative Industries                  BUSINESSES    0.059852
               Creative Industries                gva_per_hour    0.042862
                           Defence                   emp_share    0.358156
                           Defence 

In [13]:
# Summary for classification - businesses

print("\n── TOP 5 FEATURES por sector (classification lq_bus) ──────────")
top_fi_class_lqbus = (
    fi_df[(fi_df["dep_var"] == "lq_bus_binary") & (fi_df["model_type"] == "classification")]
    .sort_values(["sector", "importance"], ascending=[True, False])
    .groupby("sector")
    .head(5)[["sector", "feature", "importance"]]
    .sort_values(["sector", "importance"], ascending=[True, False])
)
print(top_fi_class_lqbus.to_string(index=False))


── TOP 5 FEATURES por sector (classification lq_bus) ──────────
                            sector                     feature  importance
            Advanced Manufacturing       apprenticeship_starts    0.119348
            Advanced Manufacturing             related_variety    0.117204
            Advanced Manufacturing     level_3+_qualifications    0.105923
            Advanced Manufacturing apprenticeship_achievements    0.089452
            Advanced Manufacturing            4g_area_coverage    0.059506
               Creative Industries       apprenticeship_starts    0.256283
               Creative Industries apprenticeship_achievements    0.156700
               Creative Industries     level_3+_qualifications    0.127495
               Creative Industries              gcse_by_age_19    0.107167
               Creative Industries           cycle_to_employer    0.051688
                           Defence       apprenticeship_starts    0.226711
                           Defence 

# For growth

In [13]:
# Summary regresion growth_emp

print("\n── TOP 5 FEATURES por sector (regression growth_emp) ──────────")
top_fi_gr_emp = (
    fi_df[(fi_df["dep_var"] == "growth_emp") & (fi_df["model_type"] == "regression")]
    .sort_values(["sector", "importance"], ascending=[True, False])
    .groupby("sector")
    .head(5)[["sector", "feature", "importance"]]
    .sort_values(["sector", "importance"], ascending=[True, False])
)
print(top_fi_gr_emp.to_string(index=False))


── TOP 5 FEATURES por sector (regression growth_emp) ──────────
                            sector                     feature  importance
            Advanced Manufacturing                    cagr_emp    0.993613
            Advanced Manufacturing          active_enterprises    0.000943
            Advanced Manufacturing                     smokers    0.000866
            Advanced Manufacturing           drive_to_employer    0.000653
            Advanced Manufacturing                  worthwhile    0.000464
               Creative Industries                    cagr_emp    0.997252
               Creative Industries                    cagr_bus    0.000634
               Creative Industries               net_additions    0.000159
               Creative Industries fe_and_skills_participation    0.000155
               Creative Industries            size_micro_share    0.000155
                           Defence                    cagr_emp    0.801759
                           Defence 

In [14]:
# Summary regresion growth_bus

print("\n── TOP 5 FEATURES por sector (regression growth_bus) ──────────")
top_fi_gr_bus = (
    fi_df[(fi_df["dep_var"] == "growth_bus") & (fi_df["model_type"] == "regression")]
    .sort_values(["sector", "importance"], ascending=[True, False])
    .groupby("sector")
    .head(5)[["sector", "feature", "importance"]]
    .sort_values(["sector", "importance"], ascending=[True, False])
)
print(top_fi_gr_bus.to_string(index=False))


── TOP 5 FEATURES por sector (regression growth_bus) ──────────
                            sector                 feature  importance
            Advanced Manufacturing                cagr_bus    0.985673
            Advanced Manufacturing                cagr_emp    0.004703
            Advanced Manufacturing               happiness    0.002193
            Advanced Manufacturing            gva_per_hour    0.000985
            Advanced Manufacturing           net_additions    0.000622
               Creative Industries                cagr_bus    0.996617
               Creative Industries  broadband_availability    0.000519
               Creative Industries        size_micro_share    0.000319
               Creative Industries         employment_rate    0.000314
               Creative Industries   apprenticeship_starts    0.000266
          Digital and Technologies                cagr_bus    0.992250
          Digital and Technologies          ks2_attainment    0.001611
          Di

In [15]:
# Summary classification growth_emp

print("\n── TOP 5 FEATURES por sector (regression growth_emp) ──────────")
top_fi_class_gremp = (
    fi_df[(fi_df["dep_var"] == "growth_emp_binary") & (fi_df["model_type"] == "classification")]
    .sort_values(["sector", "importance"], ascending=[True, False])
    .groupby("sector")
    .head(5)[["sector", "feature", "importance"]]
    .sort_values(["sector", "importance"], ascending=[True, False])
)
print(top_fi_class_gremp.to_string(index=False))


── TOP 5 FEATURES por sector (regression growth_emp) ──────────
                            sector                      feature  importance
            Advanced Manufacturing            unemployment_rate    0.082772
            Advanced Manufacturing           active_enterprises    0.076717
            Advanced Manufacturing       broadband_availability    0.074958
            Advanced Manufacturing              new_enterprises    0.068172
            Advanced Manufacturing        deaths_of_enterprises    0.065588
               Creative Industries              related_variety    0.077279
               Creative Industries       broadband_availability    0.064726
               Creative Industries public_transport_to_employer    0.064311
               Creative Industries      level_3+_qualifications    0.063775
               Creative Industries            cycle_to_employer    0.063417
                           Defence               gcse_by_age_19    0.101432
                       

In [15]:
# Summary classification growth_emp

print("\n── TOP 5 FEATURES por sector (regression growth_bus) ──────────")
top_fi_class_grbus = (
    fi_df[(fi_df["dep_var"] == "growth_bus_binary") & (fi_df["model_type"] == "classification")]
    .sort_values(["sector", "importance"], ascending=[True, False])
    .groupby("sector")
    .head(5)[["sector", "feature", "importance"]]
    .sort_values(["sector", "importance"], ascending=[True, False])
)
print(top_fi_class_grbus.to_string(index=False))


── TOP 5 FEATURES por sector (regression growth_bus) ──────────
                            sector                 feature  importance
            Advanced Manufacturing                cagr_bus    0.576508
            Advanced Manufacturing                cagr_emp    0.026897
            Advanced Manufacturing         related_variety    0.024518
            Advanced Manufacturing               happiness    0.022535
            Advanced Manufacturing          year_6_obesity    0.017217
               Creative Industries                cagr_bus    0.561851
               Creative Industries         related_variety    0.033471
               Creative Industries                cagr_emp    0.030820
               Creative Industries              weekly_pay    0.023815
               Creative Industries            gva_per_hour    0.015617
          Digital and Technologies                cagr_bus    0.593319
          Digital and Technologies         related_variety    0.035214
          Di

# Export Comparative tables

In [17]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Configuración ──────────────────────────────────────────────
OUTPUT_FILE = "feature_importance_por_sector_full_data.xlsx"

DEP_VARS  = ["lq_emp", "lq_bus", "growth_emp", "growth_bus"]
MOD_TYPES = ["regression", "classifier"]
TOP_N     = 5

# Colores por model_type
COLORS = {
    "regression": {"header": "1F4E79", "subheader": "BDD7EE", "alt": "EBF3FB"},
    "classifier": {"header": "375623", "subheader": "C6EFCE", "alt": "EBF5EB"},
}

thin  = Side(style="thin",   color="CCCCCC")
thick = Side(style="medium", color="888888")
BORDER     = Border(left=thin,  right=thin,  top=thin,  bottom=thin)
BORDER_BOT = Border(left=thin,  right=thin,  top=thin,  bottom=thick)


# ── Helpers ────────────────────────────────────────────────────
def style_cell(cell, bold=False, bg=None, font_color="000000",
               align="center", wrap=False, border=BORDER, size=10):
    if bg:
        cell.fill = PatternFill("solid", fgColor=bg)
    cell.font      = Font(bold=bold, color=font_color, name="Calibri", size=size)
    cell.alignment = Alignment(horizontal=align, vertical="center", wrap_text=wrap)
    cell.border    = border


def build_pivot(fi_df, model_type, sector):
    """
    Devuelve un DataFrame con:
      - índice: todas las features que aparecen en el top 5
                de cualquier dep_var para este sector/model_type
      - columnas: DEP_VARS
      - valores: importancia si está en el top 5, NaN si no
    """
    mask = (
        (fi_df["model_type"] == model_type) &
        (fi_df["sector"]     == sector)
    )
    sub = fi_df[mask].copy()

    # Filtrar solo top 5 por dep_var
    top = (
        sub.sort_values("importance", ascending=False)
           .groupby("dep_var")
           .head(TOP_N)
    )

    if top.empty:
        return pd.DataFrame(columns=DEP_VARS)

    pivot = (
        top.pivot_table(
            index="feature",
            columns="dep_var",
            values="importance",
            aggfunc="first"
        )
        .reindex(columns=[d for d in DEP_VARS if d in top["dep_var"].unique()])
    )

    # Ordenar features por importancia máxima entre dep_vars
    pivot["_max"] = pivot.max(axis=1)
    pivot = pivot.sort_values("_max", ascending=False).drop(columns="_max")

    return pivot


# ── Escritura de una tabla (regression o classifier) ──────────
def write_table(ws, pivot, model_type, start_row, start_col=1):
    c  = COLORS[model_type]
    hc, shc, alt = c["header"], c["subheader"], c["alt"]

    dep_vars_present = list(pivot.columns)
    n_cols = 1 + len(dep_vars_present)   # feature col + dep_var cols

    # ── Fila 1: título del modelo ──────────────────────────────
    end_col = start_col + n_cols - 1
    ws.merge_cells(
        start_row=start_row, start_column=start_col,
        end_row=start_row,   end_column=end_col
    )
    title = ws.cell(start_row, start_col, model_type.upper())
    style_cell(title, bold=True, bg=hc, font_color="FFFFFF",
               align="center", size=11)
    ws.row_dimensions[start_row].height = 20

    # ── Fila 2: headers de columnas ────────────────────────────
    h_row = start_row + 1
    feat_h = ws.cell(h_row, start_col, "Feature")
    style_cell(feat_h, bold=True, bg=shc, align="left")
    ws.row_dimensions[h_row].height = 16

    for j, dv in enumerate(dep_vars_present):
        dv_h = ws.cell(h_row, start_col + 1 + j, dv)
        style_cell(dv_h, bold=True, bg=shc, align="center")

    # ── Filas de datos ─────────────────────────────────────────
    for r_idx, (feat, row) in enumerate(pivot.iterrows()):
        data_row = h_row + 1 + r_idx
        bg = alt if r_idx % 2 == 0 else "FFFFFF"
        ws.row_dimensions[data_row].height = 15

        f_cell = ws.cell(data_row, start_col, feat)
        style_cell(f_cell, bg=bg, align="left")

        for j, dv in enumerate(dep_vars_present):
            val = row.get(dv)
            cell = ws.cell(data_row, start_col + 1 + j)
            if pd.notna(val):
                cell.value = round(float(val), 4)
                cell.number_format = "0.0000"
                style_cell(cell, bg=bg, align="center")
            else:
                cell.value = ""
                style_cell(cell, bg="F0F0F0", align="center")

    # Devuelve la fila donde terminó la tabla
    last_row = h_row + len(pivot)
    return last_row


# ── Exportación principal ──────────────────────────────────────
def export_feature_importance(fi_df, output_file=OUTPUT_FILE):
    sectors = sorted(fi_df["sector"].unique())

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for sector in sectors:
            sheet_name = str(sector)[:31]
            ws = writer.book.create_sheet(title=sheet_name)
            ws.sheet_view.showGridLines = False

            # ── Título del sector ──────────────────────────────
            ws.merge_cells("A1:F1")
            t = ws["A1"]
            t.value = f"Sector: {sector}"
            style_cell(t, bold=True, bg="2E2E2E", font_color="FFFFFF",
                       align="center", size=13)
            ws.row_dimensions[1].height = 26

            current_row = 3   # espacio tras el título

            # ── Una tabla por model_type, apiladas verticalmente ──
            for mt in MOD_TYPES:
                pivot = build_pivot(fi_df, mt, sector)

                if pivot.empty:
                    ws.cell(current_row, 1,
                            f"Sin datos para {mt} en este sector")
                    current_row += 2
                    continue

                last = write_table(ws, pivot, mt,
                                   start_row=current_row, start_col=1)
                current_row = last + 3   # espacio entre tablas

            # ── Anchos de columna ──────────────────────────────
            ws.column_dimensions["A"].width = 30
            for j in range(len(DEP_VARS)):
                ws.column_dimensions[get_column_letter(2 + j)].width = 14

        # Eliminar hoja vacía por defecto
        if "Sheet" in writer.book.sheetnames:
            del writer.book["Sheet"]

    print(f"✅ Exportado: {output_file}  ({len(sectors)} sectores)")


# ── Llamada ────────────────────────────────────────────────────
export_feature_importance(fi_df)

✅ Exportado: feature_importance_por_sector_full_data.xlsx  (7 sectores)


In [38]:
reg  = pd.read_csv("rf_comparison_regression.csv")
clf  = pd.read_csv("rf_comparison_classification.csv")

print("── REGRESIÓN ──")
print(reg[["sector","dep_var","R2_mean_full","R2_mean_theory","R2_winner"]].to_string(index=False))

print("\n── CLASIFICACIÓN ──")
print(clf[["sector","dep_var","F1_mean_full","F1_mean_theory","F1_winner"]].to_string(index=False))

── REGRESIÓN ──
                            sector  dep_var  R2_mean_full  R2_mean_theory R2_winner
            Advanced Manufacturing cagr_bus        0.0937          0.0649      full
            Advanced Manufacturing cagr_emp       -0.0788         -0.0885      full
            Advanced Manufacturing   lq_bus        0.4266          0.3645      full
            Advanced Manufacturing   lq_emp        0.2354          0.1101      full
               Creative Industries cagr_bus        0.0875          0.1053    theory
               Creative Industries cagr_emp       -0.0117         -0.0256      full
               Creative Industries   lq_bus        0.7470          0.7396      full
               Creative Industries   lq_emp        0.6433          0.3186      full
                           Defence cagr_emp       -0.2438         -0.2647      full
                           Defence   lq_bus       -0.0019         -0.0021      full
                           Defence   lq_emp      -30.5146   

In [39]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ─────────────────────────────────────────────
# DATOS
# ─────────────────────────────────────────────
reg = pd.read_csv("rf_comparison_regression.csv")
clf = pd.read_csv("rf_comparison_classification.csv")

# ─────────────────────────────────────────────
# UMBRALES Y COLORES
# ─────────────────────────────────────────────
# Regresión R²
def r2_fill(val):
    if pd.isna(val):
        return PatternFill("solid", fgColor="D9D9D9")  # gris
    if val >= 0.30:
        return PatternFill("solid", fgColor="C6EFCE")  # verde
    if val >= 0.10:
        return PatternFill("solid", fgColor="FFEB9C")  # amarillo
    return PatternFill("solid", fgColor="FFC7CE")      # rojo

# Clasificación F1
def f1_fill(val):
    if pd.isna(val):
        return PatternFill("solid", fgColor="D9D9D9")
    if val >= 0.65:
        return PatternFill("solid", fgColor="C6EFCE")  # verde
    if val >= 0.50:
        return PatternFill("solid", fgColor="FFEB9C")  # amarillo
    return PatternFill("solid", fgColor="FFC7CE")      # rojo

# Estilos generales
bold          = Font(bold=True)
white_bold    = Font(bold=True, color="FFFFFF")
header_fill   = PatternFill("solid", fgColor="2F5496")
section_fill  = PatternFill("solid", fgColor="D9E1F2")
center        = Alignment(horizontal="center", vertical="center", wrap_text=True)
left          = Alignment(horizontal="left",   vertical="center")
thin_border   = Border(
    left=Side(style="thin"), right=Side(style="thin"),
    top=Side(style="thin"),  bottom=Side(style="thin")
)

# ─────────────────────────────────────────────
# HELPER: escribir celda con estilo
# ─────────────────────────────────────────────
def write_cell(ws, row, col, value, font=None, fill=None, alignment=None, border=None):
    cell = ws.cell(row=row, column=col, value=value)
    if font:      cell.font      = font
    if fill:      cell.fill      = fill
    if alignment: cell.alignment = alignment
    if border:    cell.border    = border
    return cell

# ─────────────────────────────────────────────
# CONSTRUIR EXCEL
# ─────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)  # quitar hoja vacía default

sectors = sorted(reg["sector"].unique())

for sector in sectors:
    ws = wb.create_sheet(title=sector[:31])

    reg_s = reg[reg["sector"] == sector].copy()
    clf_s = clf[clf["sector"] == sector].copy()

    # ── TÍTULO ──────────────────────────────
    ws.merge_cells("A1:E1")
    write_cell(ws, 1, 1, f"Sector: {sector}",
               font=Font(bold=True, size=13, color="FFFFFF"),
               fill=PatternFill("solid", fgColor="1F3864"),
               alignment=center)
    ws.row_dimensions[1].height = 22

    # ── LEYENDA ─────────────────────────────
    ws.merge_cells("A2:E2")
    write_cell(ws, 2, 1,
               "🟢 Bueno (R²≥0.30 / F1≥0.65)   🟡 Marginal (R²0.10–0.29 / F1 0.50–0.64)   🔴 No recomendado",
               font=Font(italic=True, size=9),
               alignment=center)
    ws.row_dimensions[2].height = 18

    current_row = 3

    # ════════════════════════════════════════
    # BLOQUE REGRESIÓN
    # ════════════════════════════════════════
    ws.merge_cells(f"A{current_row}:E{current_row}")
    write_cell(ws, current_row, 1, "REGRESIÓN — R² (CV medio)",
               font=white_bold, fill=header_fill, alignment=center)
    ws.row_dimensions[current_row].height = 18
    current_row += 1

    # Encabezados regresión
    reg_headers = ["Variable dependiente", "R² Full", "R² Theory", "Mejor grupo", "Valoración"]
    for col_idx, h in enumerate(reg_headers, start=1):
        write_cell(ws, current_row, col_idx, h,
                   font=bold, fill=section_fill,
                   alignment=center, border=thin_border)
    ws.row_dimensions[current_row].height = 16
    current_row += 1

    # Filas regresión
    for _, row_data in reg_s.iterrows():
        dep      = row_data["dep_var"]
        r2_full  = row_data["R2_mean_full"]
        r2_theory= row_data["R2_mean_theory"]
        winner   = row_data["R2_winner"]
        best_r2  = max(r2_full, r2_theory)

        # Valoración textual
        if best_r2 >= 0.30:
            valoracion = "✅ Recomendado"
        elif best_r2 >= 0.10:
            valoracion = "⚠️ Marginal"
        else:
            valoracion = "❌ No usar"

        write_cell(ws, current_row, 1, dep,        alignment=left,   border=thin_border)
        write_cell(ws, current_row, 2, round(r2_full,   4) if pd.notna(r2_full)   else "—",
                   fill=r2_fill(r2_full),   alignment=center, border=thin_border)
        write_cell(ws, current_row, 3, round(r2_theory, 4) if pd.notna(r2_theory) else "—",
                   fill=r2_fill(r2_theory), alignment=center, border=thin_border)
        write_cell(ws, current_row, 4, winner,     alignment=center, border=thin_border)
        write_cell(ws, current_row, 5, valoracion,
                   fill=r2_fill(best_r2),   alignment=center, border=thin_border)
        current_row += 1

    current_row += 1  # espacio

    # ════════════════════════════════════════
    # BLOQUE CLASIFICACIÓN
    # ════════════════════════════════════════
    ws.merge_cells(f"A{current_row}:E{current_row}")
    write_cell(ws, current_row, 1, "CLASIFICACIÓN — F1 (CV medio)",
               font=white_bold, fill=header_fill, alignment=center)
    ws.row_dimensions[current_row].height = 18
    current_row += 1

    # Encabezados clasificación
    clf_headers = ["Variable dependiente", "F1 Full", "F1 Theory", "Mejor grupo", "Valoración"]
    for col_idx, h in enumerate(clf_headers, start=1):
        write_cell(ws, current_row, col_idx, h,
                   font=bold, fill=section_fill,
                   alignment=center, border=thin_border)
    ws.row_dimensions[current_row].height = 16
    current_row += 1

    # Filas clasificación
    for _, row_data in clf_s.iterrows():
        dep      = row_data["dep_var"]
        f1_full  = row_data["F1_mean_full"]
        f1_theory= row_data["F1_mean_theory"]
        winner   = row_data["F1_winner"]
        best_f1  = max(f1_full, f1_theory)

        if best_f1 >= 0.65:
            valoracion = "✅ Recomendado"
        elif best_f1 >= 0.50:
            valoracion = "⚠️ Marginal"
        else:
            valoracion = "❌ No usar"

        write_cell(ws, current_row, 1, dep,         alignment=left,   border=thin_border)
        write_cell(ws, current_row, 2, round(f1_full,   4) if pd.notna(f1_full)   else "—",
                   fill=f1_fill(f1_full),   alignment=center, border=thin_border)
        write_cell(ws, current_row, 3, round(f1_theory, 4) if pd.notna(f1_theory) else "—",
                   fill=f1_fill(f1_theory), alignment=center, border=thin_border)
        write_cell(ws, current_row, 4, winner,      alignment=center, border=thin_border)
        write_cell(ws, current_row, 5, valoracion,
                   fill=f1_fill(best_f1),   alignment=center, border=thin_border)
        current_row += 1

    # ── Ajustar anchos ───────────────────────
    col_widths = [32, 12, 12, 14, 18]
    for i, w in enumerate(col_widths, start=1):
        ws.column_dimensions[get_column_letter(i)].width = w

wb.save("rf_model_quality_summary.xlsx")
print("✅ Exportado: rf_model_quality_summary.xlsx")


✅ Exportado: rf_model_quality_summary.xlsx


In [44]:
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ─────────────────────────────────────────────
# CODEBOOK
# ─────────────────────────────────────────────
cb = pd.read_excel("PP422_Variable_Codebook.xlsx")
cb = cb[["variable", "group", "definition"]].dropna(subset=["variable"])
cb["variable"] = cb["variable"].str.strip()
codebook = cb.set_index("variable").to_dict(orient="index")

def get_meta(feature, field):
    entry = codebook.get(feature.strip(), {})
    return entry.get(field, "—")

# ─────────────────────────────────────────────
# UMBRALES
# ─────────────────────────────────────────────
R2_GOOD     = 0.30
R2_MARGINAL = 0.10
F1_GOOD     = 0.65
F1_MARGINAL = 0.50
CUM_THRESH  = 0.80
MIN_FEAT    = 2
MAX_FEAT    = 5

# ─────────────────────────────────────────────
# MÉTRICAS
# ─────────────────────────────────────────────
reg = pd.read_csv("rf_comparison_regression.csv")
clf = pd.read_csv("rf_comparison_classification.csv")

reg["R2_best"] = reg[["R2_mean_full", "R2_mean_theory"]].max(axis=1)
clf["F1_best"] = clf[["F1_mean_full", "F1_mean_theory"]].max(axis=1)

# ─────────────────────────────────────────────
# FEATURE IMPORTANCE
# ─────────────────────────────────────────────
fi = fi_df[fi_df["group"] == "theory"].copy()

def get_top_features(fi_data, sector, dep_var,
                     cum_thresh=CUM_THRESH, min_f=MIN_FEAT, max_f=MAX_FEAT):
    sub = fi_data[
        (fi_data["sector"]  == sector) &
        (fi_data["dep_var"] == dep_var)
    ].sort_values("importance", ascending=False).reset_index(drop=True)
    if sub.empty:
        return pd.DataFrame()
    sub["cumulative"] = sub["importance"].cumsum()
    n = int((sub["cumulative"] < cum_thresh).sum()) + 1
    n = max(min_f, min(n, max_f))
    return sub[["feature", "importance"]].head(n).reset_index(drop=True)

# ─────────────────────────────────────────────
# ESTILOS
# ─────────────────────────────────────────────
FILLS = {
    "good":    PatternFill("solid", fgColor="C6EFCE"),
    "marginal":PatternFill("solid", fgColor="FFEB9C"),
    "bad":     PatternFill("solid", fgColor="FFC7CE"),
    "header":  PatternFill("solid", fgColor="2F5496"),
    "section": PatternFill("solid", fgColor="D9E1F2"),
    "title":   PatternFill("solid", fgColor="1F3864"),
    "feat_bg": PatternFill("solid", fgColor="F2F2F2"),
}

def quality_fill(val, metric="r2"):
    good, marg = (R2_GOOD, R2_MARGINAL) if metric == "r2" else (F1_GOOD, F1_MARGINAL)
    if pd.isna(val) or val < marg: return FILLS["bad"]
    if val < good:                 return FILLS["marginal"]
    return FILLS["good"]

def quality_label(val, metric="r2"):
    good, marg = (R2_GOOD, R2_MARGINAL) if metric == "r2" else (F1_GOOD, F1_MARGINAL)
    if pd.isna(val) or val < marg: return "❌ Not recommended"
    if val < good:                 return "⚠️ Marginal"
    return "✅ Recommended"

thin   = Side(style="thin")
BORDER = Border(left=thin, right=thin, top=thin, bottom=thin)
CENTER = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT   = Alignment(horizontal="left",   vertical="center", wrap_text=True)
BOLD   = Font(bold=True)
WHITE_BOLD = Font(bold=True, color="FFFFFF")

def wc(ws, row, col, value, font=None, fill=None, align=None):
    c = ws.cell(row=row, column=col, value=value)
    if font:  c.font      = font
    if fill:  c.fill      = fill
    if align: c.alignment = align
    c.border = BORDER
    return c

# ─────────────────────────────────────────────
# COLUMNAS
# ─────────────────────────────────────────────
HEADERS_REG = ["Dependent Variable", "R² (best)", "Quality",
               "Feature", "Importance", "Cumul.", "Group", "Definition"]
HEADERS_CLF = ["Dependent Variable", "F1 (best)", "Quality",
               "Feature", "Importance", "Cumul.", "Group", "Definition"]
NCOLS = 8

REG_DEPS = ["lq_emp", "lq_bus"]
CLF_DEPS = ["lq_emp_binary", "lq_bus_binary",
            "cagr_emp_binary", "cagr_bus_binary"]

# ─────────────────────────────────────────────
# HELPER: fila de warning
# ─────────────────────────────────────────────
def write_warning_row(ws, current_row, dep, metric_val, fill_q, label, msg):
    """Escribe fila de warning: celdas individuales primero, merge después."""
    # 1. Escribir TODAS las celdas primero
    wc(ws, current_row, 1, dep,   align=LEFT)
    wc(ws, current_row, 2,
       round(float(metric_val), 4) if pd.notna(metric_val) else "—",
       fill=fill_q, align=CENTER)
    wc(ws, current_row, 3, label, fill=fill_q, align=CENTER)
    wc(ws, current_row, 4, msg,
       font=Font(italic=True, color="9C0006"),
       fill=FILLS["bad"], align=LEFT)
    for col in range(5, NCOLS + 1):
        wc(ws, current_row, col, "", fill=FILLS["bad"])
    # 2. Merge D:H después de escribir
    ws.merge_cells(
        start_row=current_row, start_column=4,
        end_row=current_row,   end_column=NCOLS
    )
    return current_row + 1

# ─────────────────────────────────────────────
# HELPER: bloque de features
# ─────────────────────────────────────────────
def write_features_block(ws, current_row, dep, metric_val,
                         fill_q, label, top):
    first_row = current_row
    for i, feat_row in top.iterrows():
        feat = feat_row["feature"]
        imp  = feat_row["importance"]
        cum  = top["importance"].iloc[:i+1].sum()
        grp  = get_meta(feat, "group")
        defn = get_meta(feat, "definition")

        # Columnas identificadoras (solo en primera fila)
        if i == 0:
            wc(ws, current_row, 1, dep,
               align=LEFT)
            wc(ws, current_row, 2,
               round(float(metric_val), 4),
               fill=fill_q, align=CENTER)
            wc(ws, current_row, 3, label,
               fill=fill_q, align=CENTER)
        else:
            wc(ws, current_row, 1, "", align=LEFT)
            wc(ws, current_row, 2, "", fill=fill_q, align=CENTER)
            wc(ws, current_row, 3, "", fill=fill_q, align=CENTER)

        # Feature columns
        wc(ws, current_row, 4, feat,
           fill=FILLS["feat_bg"], align=LEFT)
        wc(ws, current_row, 5,
           round(float(imp), 4),
           fill=FILLS["feat_bg"], align=CENTER)
        wc(ws, current_row, 6,
           f"{cum:.1%}",
           fill=FILLS["feat_bg"], align=CENTER)
        wc(ws, current_row, 7, grp,
           fill=FILLS["feat_bg"], align=LEFT)
        wc(ws, current_row, 8, defn,
           fill=FILLS["feat_bg"], align=LEFT)

        current_row += 1

    # Merge primeras 3 columnas si hay varias features
    if len(top) > 1:
        for col in [1, 2, 3]:
            ws.merge_cells(
                start_row=first_row, start_column=col,
                end_row=current_row - 1, end_column=col
            )
            c = ws.cell(first_row, col)
            c.alignment = LEFT if col == 1 else CENTER
            if col in [2, 3]:
                c.fill = fill_q
            c.border = BORDER

    return current_row

# ─────────────────────────────────────────────
# CONSTRUIR EXCEL
# ─────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)

sectors = sorted(fi["sector"].unique())

for sector in sectors:
    ws = wb.create_sheet(title=sector[:31])

    # ── Título ──────────────────────────────
    ws.merge_cells(f"A1:{get_column_letter(NCOLS)}1")
    wc(ws, 1, 1, f"Sector: {sector}",
       font=Font(bold=True, size=13, color="FFFFFF"),
       fill=FILLS["title"], align=CENTER)
    ws.row_dimensions[1].height = 24

    # ── Leyenda ─────────────────────────────
    ws.merge_cells(f"A2:{get_column_letter(NCOLS)}2")
    wc(ws, 2, 1,
       "🟢 Good (R²≥0.30 / F1≥0.65)   "
       "🟡 Marginal (R²0.10–0.29 / F1 0.50–0.64)   "
       "🔴 Not recommended  |  "
       "Features: top 80% cumulative importance (min 2, max 5)",
       font=Font(italic=True, size=9), align=CENTER)
    ws.row_dimensions[2].height = 16

    current_row = 3

    # ════════════════════════════════════════
    # BLOQUE REGRESIÓN
    # ════════════════════════════════════════
    ws.merge_cells(f"A{current_row}:{get_column_letter(NCOLS)}{current_row}")
    wc(ws, current_row, 1, "REGRESSION — Location Quotient (LQ)",
       font=WHITE_BOLD, fill=FILLS["header"], align=CENTER)
    ws.row_dimensions[current_row].height = 18
    current_row += 1

    # Aviso CAGR
    ws.merge_cells(f"A{current_row}:{get_column_letter(NCOLS)}{current_row}")
    wc(ws, current_row, 1,
       "⚠️  CAGR (continuous growth) not shown in regression "
       "— R² ≈ 0 across all sectors. Use binary classification instead.",
       font=Font(italic=True, size=9, color="9C0006"),
       fill=FILLS["bad"], align=CENTER)
    ws.row_dimensions[current_row].height = 16
    current_row += 1

    # Encabezados
    for col, h in enumerate(HEADERS_REG, start=1):
        wc(ws, current_row, col, h,
           font=BOLD, fill=FILLS["section"], align=CENTER)
    ws.row_dimensions[current_row].height = 16
    current_row += 1

    for dep in REG_DEPS:
        r2_row = reg[(reg["sector"] == sector) & (reg["dep_var"] == dep)]
        r2_val = r2_row["R2_best"].values[0] if not r2_row.empty else np.nan
        fill_q = quality_fill(r2_val, "r2")
        label  = quality_label(r2_val, "r2")

        if pd.isna(r2_val) or r2_val < R2_MARGINAL:
            current_row = write_warning_row(
                ws, current_row, dep, r2_val, fill_q, label,
                "❌ Insufficient R² — features not shown"
            )
        else:
            top = get_top_features(fi, sector, dep)
            if not top.empty:
                current_row = write_features_block(
                    ws, current_row, dep, r2_val, fill_q, label, top
                )
            else:
                current_row = write_warning_row(
                    ws, current_row, dep, r2_val, fill_q, label,
                    "⚠️ No feature importance data available"
                )
        current_row += 1  # espacio entre dep_vars

    current_row += 1

    # ════════════════════════════════════════
    # BLOQUE CLASIFICACIÓN
    # ════════════════════════════════════════
    ws.merge_cells(f"A{current_row}:{get_column_letter(NCOLS)}{current_row}")
    wc(ws, current_row, 1, "BINARY CLASSIFICATION — LQ and CAGR",
       font=WHITE_BOLD, fill=FILLS["header"], align=CENTER)
    ws.row_dimensions[current_row].height = 18
    current_row += 1

    # Encabezados
    for col, h in enumerate(HEADERS_CLF, start=1):
        wc(ws, current_row, col, h,
           font=BOLD, fill=FILLS["section"], align=CENTER)
    ws.row_dimensions[current_row].height = 16
    current_row += 1

    for dep in CLF_DEPS:
        f1_row = clf[(clf["sector"] == sector) & (clf["dep_var"] == dep)]
        f1_val = f1_row["F1_best"].values[0] if not f1_row.empty else np.nan
        fill_q = quality_fill(f1_val, "f1")
        label  = quality_label(f1_val, "f1")
        dep_fi = dep.replace("_binary", "")

        if pd.isna(f1_val) or f1_val < F1_MARGINAL:
            current_row = write_warning_row(
                ws, current_row, dep, f1_val, fill_q, label,
                "❌ Insufficient F1 — features not shown"
            )
        else:
            top = get_top_features(fi, sector, dep_fi)
            if not top.empty:
                current_row = write_features_block(
                    ws, current_row, dep, f1_val, fill_q, label, top
                )
            else:
                current_row = write_warning_row(
                    ws, current_row, dep, f1_val, fill_q, label,
                    "⚠️ No feature importance data available"
                )
        current_row += 1  # espacio entre dep_vars

    # ── Ajustar anchos ───────────────────────
    for i, w in enumerate([22, 11, 18, 26, 12, 9, 26, 55], start=1):
        ws.column_dimensions[get_column_letter(i)].width = w

    # Altura para filas con definición larga
    for row in ws.iter_rows():
        for cell in row:
            if cell.column == 8 and cell.value and len(str(cell.value)) > 80:
                ws.row_dimensions[cell.row].height = 45

wb.save("rf_feature_importance_filtered.xlsx")
print("✅ Exported: rf_feature_importance_filtered.xlsx")

✅ Exported: rf_feature_importance_filtered.xlsx
